# Choisir un modèle emnedding

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  
api_key = os.getenv("API_KEY")

In [ ]:
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=api_key)



In [ ]:
text1 = "Erreur lors du chargement du tableau de bord"
text2 = "Le tableau de bord ne s'affiche pas"
text3 = "Le client a commandé un nouveau serveur"

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialiser le modèle
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Texte à transformer en vecteur
texte = "Les embeddings transforment les textes en vecteurs numériques."

# Générer l'embedding
vecteur = embeddings.embed_query(texte)

# Afficher le résultat
print("Vecteur :", vecteur)
print("Nombre de dimensions :", len(vecteur))

In [ ]:
vecteur1 = embeddings.embed_query(text1)
vecteur2 = embeddings.embed_query(text2)    
vecteur3 = embeddings.embed_query(text3)

print("Vecteur 1 :", vecteur1)
print("Vecteur 2 :", vecteur2)
print("Vecteur 3 :", vecteur3)

In [ ]:
import numpy as np
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("text1 :", text1)
print("text2 :", text2)
print("text3 :", text3)

print("Similarité entre vecteur 1 et vecteur 2 :", cosine_similarity(vecteur1, vecteur2))
print("Similarité entre vecteur 2 et vecteur 3 :", cosine_similarity(vecteur2, vecteur3))
print("Similarité entre vecteur 3 et vecteur 1 :", cosine_similarity(vecteur3, vecteur1))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos 

cos([vecteur1], [vecteur3])[0][0]

In [ ]:
print("Similarité entre vecteur 1 et vecteur 3 :", cos([vecteur1], [vecteur2])[0][0])
print("Similarité entre vecteur 1 et vecteur 2 :", cos([vecteur1], [vecteur3])[0][0])
print("Similarité entre vecteur 2 et vecteur 3 :", cos([vecteur2], [vecteur3])[0][0])

In [ ]:
# Visualiser les embeddinags avec t-SNE et umap
sentences = [
    "Erreur lors du chargement du tableau de bord.",
    "Le tableau de bord ne s'affiche pas.",
    "Impossible de se connecter à l'API.",
    "Problème d'authentification sur le service.",
    "Le client a commandé un nouveau serveur.",
    "Livraison prévue pour la semaine prochaine.",
    "Le serveur physique est tombé en panne.",
    "Mise à jour réussie du système."
]

vectors = [embeddings.embed_query(sentence) for sentence in sentences]


In [ ]:
from sklearn.manifold import TSNE
import numpy as np
   
vectors_np = np.array(vectors)
tsne = TSNE(n_components=2, perplexity=5, random_state=2)
points_tsne = tsne.fit_transform(vectors_np)


In [ ]:
import matplotlib.pyplot as plt
   
plt.figure(figsize=(8,6))
plt.scatter(points_tsne[:,0], points_tsne[:,1])
for i, txt in enumerate(sentences):
    plt.annotate(txt, (points_tsne[i,0], points_tsne[i,1]))
plt.title("Visualisation TSNE des embeddings (2D)")
plt.show()

In [ ]:
import umap
   
reducer = umap.UMAP(n_components=2, random_state=42)
points_umap = reducer.fit_transform(vectors)
   
plt.figure(figsize=(8,6))
plt.scatter(points_umap[:,0], points_umap[:,1])
   
for i, txt in enumerate(sentences):
        plt.annotate(txt, (points_umap[i,0], points_umap[i,1]))
   
plt.title("UMAP - Visualisation des embeddings (2D)")
plt.show()

In [ ]:
from sentence_transformers import SentenceTransformer
model_local = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
text_local = "la dashboard ne se charge plus."

embedding_local = model_local.encode(text_local)

print("Embedding local :", embedding_local[:10])  # Affiche les 10 premières dimensions de l'embedding

In [ ]:
text_local = "le tableau de bord ne se charge plus."

embedding_local = model_local.encode(text_local)

print("Embedding local :", embedding_local[:10])  # Affiche les 10 premières dimensions de l'embedding

In [ ]:
sentences = [
        "Erreur lors du chargement du tableau de bord.",
       "Le tableau de bord ne s'affiche pas.",
       "Le client a commandé un nouveau serveur."
    ]
vecs = model_local.encode(sentences)
vecs

# Chargement des documents

On charge les fichiers texte du dossier `data/text_files`. Le chemin est résolu depuis la racine du projet ou depuis le dossier `notebooks`. Chaque document garde son nom de fichier comme source.

In [ ]:
from pathlib import Path

candidats = [Path("data/text_files"), Path("../data/text_files")]
docs_directory = next((p.resolve() for p in candidats if p.is_dir()), None)
if docs_directory is None:
    raise FileNotFoundError("Dossier data/text_files introuvable depuis le répertoire courant.")

raw_docs = []
metadatas = []
for fichier in sorted(docs_directory.glob("*.txt")):
    raw_docs.append(fichier.read_text(encoding="utf-8"))
    metadatas.append({"source": fichier.name})

print(f"{len(raw_docs)} fichier(s) chargé(s) depuis {docs_directory}")

## Découper les documents en passages

Le découpage limite la taille des extraits transmis au moteur de recherche et au modèle.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""],
)
all_chunks = splitter.create_documents(raw_docs, metadatas=metadatas)
print(f"{len(all_chunks)} passage(s) créé(s)")
if all_chunks:
    print(all_chunks[0].page_content)

## Indexer les passages avec des embeddings gratuits

On réutilise le modèle local `all-MiniLM-L6-v2` déclaré plus haut. Son premier usage peut télécharger les poids du modèle. Aucune clé API n’est nécessaire pour cette partie.

In [ ]:
# Conversion en documents LangChain avec leur source conservée
from langchain_core.documents import Document

documents = [
    Document(page_content=chunk.page_content, metadata=chunk.metadata)
    for chunk in all_chunks
]

print(f"{len(documents)} document(s) prêt(s) pour l'indexation")

In [ ]:
# Vérifier la dimension d'un embedding local
if all_chunks:
    sample_vector = embeddings.embed_query(all_chunks[0].page_content)
    print("Dimension du vecteur :", len(sample_vector))

In [ ]:
# Créer la base vectorielle Chroma à partir des passages
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="support_docs_local",
    persist_directory="./chroma_support_docs_local",
)
print(f"Index Chroma créé avec {len(documents)} passage(s)")

## Rechercher les passages pertinents

La recherche renvoie les passages les plus proches de la question selon leurs embeddings.

In [ ]:
query = "Pourquoi le tableau de bord ne se charge plus ?"
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Résultat {i} ---")
    print("Source :", doc.metadata.get("source", "inconnue"))
    print(doc.page_content)

## Construire le prompt et répondre avec Ollama (facultatif)

Pour générer une réponse avec un modèle gratuit exécuté en local, installez Ollama et téléchargez un modèle, par exemple `ollama pull qwen2.5:3b`. Puis installez le connecteur avec `%pip install langchain-ollama`. Cette étape ne nécessite aucun compte ni crédit OpenAI.

In [ ]:
def construire_contexte(docs):
    return "\n\n".join(
        f"[Source : {doc.metadata.get('source', 'inconnue')}]\n{doc.page_content}"
        for doc in docs
    )

def construire_prompt(question, docs):
    contexte = construire_contexte(docs)
    return f"""Tu es un assistant technique interne.
Réponds uniquement à partir du contexte. Cite la source utilisée.
Si le contexte ne permet pas de répondre, dis : « Je ne sais pas ».

Contexte :
{contexte}

Question : {question}
Réponse :"""

In [ ]:
# Cellule facultative : nécessite Ollama en cours d'exécution et qwen2.5:3b téléchargé.
# Décommentez ces lignes après avoir installé langchain-ollama.
# from langchain_ollama import ChatOllama
# modele_local = ChatOllama(model="qwen2.5:3b", temperature=0)
# reponse = modele_local.invoke(construire_prompt(query, results))
# print(reponse.content)

## Recherche avec scores et refus si le contexte est trop éloigné

Chroma renvoie ici une distance : plus elle est petite, plus le passage est proche de la question. Le seuil doit être ajusté aux données et au modèle utilisés.

In [ ]:
query = "Quelle est la cause des retards observés dans le tableau de bord Analytics ?"
results_with_scores = vector_store.similarity_search_with_score(query, k=5)

for doc, score in results_with_scores:
    print(f"\nDistance : {score:.3f}")
    print("Source :", doc.metadata.get("source", "inconnue"))
    print(doc.page_content[:300])

THRESHOLD = 1.05
filtered_docs = [doc for doc, score in results_with_scores if score <= THRESHOLD]
if not filtered_docs:
    print("Aucun passage suffisamment proche : réponse attendue, « Je ne sais pas ». ")
else:
    print(f"{len(filtered_docs)} passage(s) retenu(s)")
    print(construire_prompt(query, filtered_docs))